In [10]:
from stable_platform_matchings.optimization.optimizer import Optimizer
from stable_platform_matchings.generation.instance_generator import InstanceGenerator
from stable_platform_matchings.domain.instance import Instance
from stable_platform_matchings.graphs.road_graphs import RoadGraph

In [11]:
import sys
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from pprint import pprint
import random

In [12]:

TEXTWIDTH = 80
SIM_SIZE = 10
N_INTS = 1

INDO_CRS = "EPSG:23867"
DATA_DIR = Path("../../SyntheticInstanceGenerator/data")

FARMERS_PATH = DATA_DIR / "farmers.csv"
FARMERS_14_PATH = DATA_DIR / "farmers_14.csv"
INTS_PATH = DATA_DIR / "intermediaries.csv"
GRAPH_PATH = DATA_DIR / "graph_0-14960_00_new.pickle"
ALPHA_PATH = DATA_DIR / "precomputed_alpha.json"
SIGMAS_PATH = DATA_DIR / "precomputed_sigmas.json"

In [13]:
with open(GRAPH_PATH, "rb") as f:
    graph = pickle.load(f)

In [14]:
ig = InstanceGenerator(
    FARMERS_PATH,
    FARMERS_14_PATH,
    INTS_PATH,
    GRAPH_PATH,
    ALPHA_PATH,
    SIGMAS_PATH
)

In [15]:
ig.gen_intermediaries(12, 12)
ig.gen_pickups(2, 2, n_cycles=5)

In [16]:
ig.pickups_df

,farmer_id,farmer_x,farmer_y,farmer_lon,farmer_lat,cycle_phase,quantity,intermediary_id,nominal_day,schedule_offset,day,scaled_quantity
1,goofy_kalam_d0_f0,892117.880185,-36182.515101,102.521687,-0.326732,0,0.9,goofy_kalam,0,1,1,0.9
2,goofy_kalam_d0_f0,892117.880185,-36182.515101,102.521687,-0.326732,0,0.9,goofy_kalam,14,2,16,0.9
3,goofy_kalam_d0_f0,892117.880185,-36182.515101,102.521687,-0.326732,0,0.9,goofy_kalam,28,0,28,0.9
4,goofy_kalam_d0_f0,892117.880185,-36182.515101,102.521687,-0.326732,0,0.9,goofy_kalam,42,0,42,0.9
5,goofy_kalam_d0_f0,892117.880185,-36182.515101,102.521687,-0.326732,0,0.9,goofy_kalam,56,4,60,0.9
...,...,...,...,...,...,...,...,...,...,...,...,...
4782,wonderful_jepsen_d13_f1,869617.880185,-29182.515101,102.319826,-0.263577,13,1.4,wonderful_jepsen,13,1,14,1.4
4783,wonderful_jepsen_d13_f1,869617.880185,-29182.515101,102.319826,-0.263577,13,1.4,wonderful_jepsen,27,3,30,1.4
4784,wonderful_jepsen_d13_f1,869617.880185,-29182.515101,102.319826,-0.263577,13,1.4,wonderful_jepsen,41,0,41,1.4
4785,wonderful_jepsen_d13_f1,869617.880185,-29182.515101,102.319826,-0.263577,13,1.4,wonderful_jepsen,55,1,56,1.4


In [17]:
instance_dict = ig.gen_instance(
    instance_id=100,
    day=69,
    n_hist_sets=3,
    dev_mode="required_only"
)

pprint(instance_dict)

platform = Instance.from_dict(instance_dict)
platform.set_graph(RoadGraph(graph))
rng = np.random.default_rng(1)

epsilon = {intermediary.id: random.randint(1, 5) for intermediary in platform.intermediaries}
het_costs = {intermediary.id: (platform.dist_to_mill[intermediary.id] * 2) for intermediary in platform.intermediaries}

parameters = {
    "epsilon": epsilon,
    "backend": "gurobi",
    "het_costs": het_costs,
    "aggregate": True,
    "pay_unmatched": False
}

opt = Optimizer(platform, parameters)

summary_vanilla = opt.solve("heuristic_optimized", options={
    "structured_farmer_payments": False,
    "domination": False,})

summary_vanilla.max_intermediary_welfare_solution.platform_profit/platform.lc_to_usd

{'farmers': [{'farmer_id': 'goofy_kalam_d13_f1',
              'intermediary_id': 'goofy_kalam',
              'location': (-0.34031397008585645, 102.4207758078138),
              'quantity': 1.2},
             {'farmer_id': 'stoic_pasteur_d2_f0',
              'intermediary_id': 'stoic_pasteur',
              'location': (-0.5660896045619844, 102.42312461245945),
              'quantity': 1.2},
             {'farmer_id': 'stoic_pasteur_d11_f1',
              'intermediary_id': 'stoic_pasteur',
              'location': (-0.5570651050239062, 102.41190550371675),
              'quantity': 1.3},
             {'farmer_id': 'stoic_pasteur_d12_f5',
              'intermediary_id': 'stoic_pasteur',
              'location': (-0.5615780447521411, 102.41639367330322),
              'quantity': 0.7},
             {'farmer_id': 'awesome_hopper_d0_f2',
              'intermediary_id': 'awesome_hopper',
              'location': (-0.3335304018579805, 102.44992802517754),
              'quantity': 

GurobiError: Version number is 13.0, license is for version 12.0